# Initial Exploratory Data Analysis: Online Retail Dataset

This notebook performs the initial exploratory data analysis for the CIND820 final project. The goal is to understand the structure, quality, and basic business meaning of the Online Retail dataset before performing detailed sales analysis, customer behaviour analysis, and RFM segmentation.

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
#upload raw excel data for analysis
df = pd.read_excel("/content/sample_data/Online_Retail_Raw_Data.xlsx")
df.head() # to check first five rows

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [5]:
# check how many rows and columns in the dataset
df.shape

(541909, 8)

* The dataset contains 541,909 rows and 8 columns. Each row represents a retail transaction record, including invoice number, stock code, product description, quantity, invoice date, unit price, customer ID, and country.

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [8]:
#check missing values in the dataset for data quality
df.isnull().sum()

,0
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


* ### Missing Values

The initial missing value check shows that the Description column has 1,454 missing values and the CustomerID column has 135,080 missing values. Missing CustomerID values are important because customer-level analysis, such as RFM segmentation, requires valid customer identifiers. These missing values will need to be handled carefully during the data cleaning stage.

In [9]:
#check duplicate record in the dataset
df.duplicated().sum()

np.int64(5268)


* The dataset contains 5,268 duplicate rows. These duplicate records will be reviewed and removed during data cleaning to avoid overstating revenue, transaction volume, and product demand.

In [10]:
#check Column names
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')

### Columns Review

The dataset contains 8 original columns: InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, and Country. These columns provide transaction-level information that can support product analysis, revenue analysis, country-level sales analysis, customer purchasing behaviour analysis, and later RFM customer segmentation.

In [11]:
#check numeric summary in the dataset
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386048,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


### Numeric Summary

The numeric summary shows that the dataset contains 541,909 transaction records. The average quantity per transaction is approximately 9.55 units, while the average unit price is approximately 4.61. However, the minimum quantity is -80,995 and the maximum quantity is 80,995. This indicates that the dataset contains return or cancellation transactions, as well as possible large-volume transactions or outliers.

The UnitPrice column also contains unusual values. The minimum UnitPrice is negative, and the maximum UnitPrice is very high. These values require further review before final analysis because they may affect revenue calculations and business interpretation.

In [12]:
#check negative quantity in the dataset
negative_quantity = df[df['Quantity'] < 0]
negative_quantity. shape

(10624, 8)

### Negative Quantity Records

The dataset contains 10,624 records where Quantity is less than zero. These records likely represent product returns, cancelled transactions, or invoice adjustments. Since negative quantities can reduce total revenue and distort product demand analysis, they will be reviewed carefully during the data cleaning stage.

For sales and demand analysis, these records may be excluded from the cleaned dataset. However, they are still important because they show that returns and cancellations exist in the business process.

In [13]:
#check negative or zero price
zero_or_negative_price = df[df['UnitPrice'] <= 0]
zero_or_negative_price.shape

(2517, 8)

### Zero or Negative Unit Price Records

The dataset contains 2,517 records where UnitPrice is zero or negative. These records may represent free items, corrections, data entry issues, or non-standard transactions. Since revenue is calculated using Quantity multiplied by UnitPrice, zero or negative prices can create misleading revenue values.

These records will be reviewed during data cleaning and may be excluded from revenue-based analysis if they do not represent valid sales transactions.

In [21]:
#check cancelled and returned orders
cancelled_orders = df[df['InvoiceNo'].str.startswith('C', na=False)]
cancelled_orders.shape

(9288, 9)

### Cancelled Orders and Returns

The dataset contains 9,288 records where InvoiceNo starts with the letter “C”. These records likely represent cancelled orders or returns. This confirms that the raw dataset includes both completed sales and reverse transactions.

For the main revenue, product demand, and customer purchasing behaviour analysis, cancelled transactions will need to be handled separately or removed from the cleaned sales dataset. This will help avoid underestimating true sales performance or misinterpreting product demand.

In [17]:
# to create revenue column
df['Revenue'] = df['Quantity'] * df['UnitPrice']
df[['Quantity','Description','UnitPrice','Revenue']].head()

,Quantity,Description,UnitPrice,Revenue
0,6,WHITE HANGING HEART T-LIGHT HOLDER,2.55,15.30
1,6,WHITE METAL LANTERN,3.39,20.34
2,8,CREAM CUPID HEARTS COAT HANGER,2.75,22.00
3,6,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,20.34
4,6,RED WOOLLY HOTTIE WHITE HEART.,3.39,20.34


### Revenue Variable Creation

A new Revenue column was created by multiplying Quantity by UnitPrice. This variable will be used as the main measure for revenue analysis, product performance analysis, country-level sales analysis, monthly trend analysis, and customer monetary value in RFM segmentation.

In [18]:
df['Revenue'].describe()

,Revenue
count,541909.000000
mean,17.987795
std,378.810824
min,-168469.600000
25%,3.400000
50%,9.750000
75%,17.400000
max,168469.600000


### Revenue Summary

The revenue summary shows that the average transaction revenue is approximately 17.99. However, the minimum revenue value is negative and the maximum revenue value is very high. This confirms that the dataset contains returns, cancellations, and possible outliers.

Before final analysis, the dataset should be cleaned by reviewing cancelled invoices, negative quantities, zero or negative prices, missing CustomerID values, and duplicate records. This will improve the reliability of the revenue analysis and customer segmentation results.

### Initial EDA Summary

The initial exploratory data analysis shows that the Online Retail dataset is suitable for retail sales and customer purchasing behaviour analysis, but it requires careful cleaning before final analysis. The dataset contains 541,909 transaction records and 8 original variables. Important data quality issues were identified, including missing CustomerID values, missing product descriptions, duplicate records, negative quantities, zero or negative unit prices, and cancelled invoices.

These issues are important because they can affect revenue totals, product demand analysis, customer segmentation, and business recommendations. Therefore, the next step will be to create a cleaned dataset by removing or separating invalid sales records, cancelled orders, duplicate rows, and records that cannot support customer-level analysis. This cleaning process will support more accurate revenue analysis, country-level performance analysis, product analysis, monthly sales trend analysis, and RFM customer segmentation.